# Optimized matched CE-only: 100-step ETA pilot

Notebook này đo thời gian thật trên Kaggle T4 cho đối chứng CE-only đã tối ưu.
Toàn bộ nhánh KD trọng số 0 được xóa khỏi đường chạy:

- không nạp Teacher;
- không đọc pseudo-caption;
- không chạy student decoder lần hai;
- không tính KL;
- không tạo dense vocabulary map;
- mỗi microbatch chỉ chạy một Student encoder và một human-caption decoder.

Objective:

\[
\mathcal{L}_{control}=\mathcal{L}_{CE}.
\]

Pilot thực hiện đúng 100 optimizer step với physical batch 1 và accumulation
64, sau đó hủy toàn bộ thay đổi. Không lưu checkpoint, không full train và
không đọc Test.


## Cell 1 — Authorization và input tối thiểu


In [1]:
import copy
import csv
import gc
import hashlib
import json
import math
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

STARTED = time.perf_counter()
SEED = 42
PILOT_STEPS = 100
PILOT_WARMUP_STEPS = 5
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
DATASET_ROOT = (
    INPUT_ROOT / "datasets"
    if (INPUT_ROOT / "datasets").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//"
    "XLKD05_Dense_Mapped_LogitKD_optimized_ce_eta_pilot",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
LOG = OUT / "logs"
RESULT = OUT / "results"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, LOG, RESULT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT = OUT
AUDIT_DIR = AUDIT
CONFIG_DIR = CONFIG
LOG_DIR = LOG
RESULT_DIR = RESULT
MANIFEST_DIR = MANIFEST


def now():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target


DISCOVERY = {}
for candidate in NOTEBOOK_ROOT.rglob("*"):
    if candidate.is_file():
        DISCOVERY.setdefault(candidate.name, []).append(candidate.resolve())


def exactly_one(name):
    matches = sorted(DISCOVERY.get(name, []))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0]


def verify_csv_manifest(root, reference):
    path = resolve_inside(root, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if "rows" in reference:
        assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows


def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError("Pilot CE-only yêu cầu Kaggle GPU T4 hoặc mạnh hơn.")
set_deterministic(SEED)
device = torch.device("cuda")

NEXT_PHASE_PATH = exactly_one(
    "XLKD05_Dense_Mapped_LogitKD_"
    "NEXT_PHASE_ENTRY_COMPLETE.json"
)
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P2A_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
S1_PATH = exactly_one("S1_SEED42_FULL_RUN_COMPLETE.json")
TREATMENT_PATH = exactly_one(
    "XLKD04_DENSE_MAPPED_LOGIT_KD_SEED42_FULL_RUN_COMPLETE.json"
)

NEXT_PHASE = read_json(NEXT_PHASE_PATH)
P1 = read_json(P1_PATH)
P2A = read_json(P2A_PATH)
S1 = read_json(S1_PATH)
TREATMENT = read_json(TREATMENT_PATH)
assert NEXT_PHASE["status"] == "complete"
assert NEXT_PHASE["reproduction_gate"] == "passed"
assert NEXT_PHASE["final_scientific_gate"] == "pending"
assert P1["status"] == "ready"
assert P2A["status"] == "ready"
assert S1["status"] == "complete"
assert S1["student_id"] == "S1" and int(S1["seed"]) == SEED
assert TREATMENT["status"] == "complete"
assert int(TREATMENT["seed"]) == SEED
for payload in [NEXT_PHASE, S1, TREATMENT]:
    assert payload["test_set_status"] == "sealed"
    assert payload["test_set_used"] is False
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
assert P2A["data_policy"]["test_set_status"] == "sealed"
assert P2A["data_policy"]["test_set_used"] is False

control_ref = NEXT_PHASE["matched_budget_contract"]
CONTROL_CONTRACT_PATH = resolve_inside(
    NEXT_PHASE_PATH.parent, control_ref["relative_path"]
)
assert sha256(CONTROL_CONTRACT_PATH) == control_ref["sha256"]
CONTROL_CONTRACT = read_json(CONTROL_CONTRACT_PATH)
assert CONTROL_CONTRACT["status"] == "locked"
assert CONTROL_CONTRACT["arms"]["control"]["loss"] == "human CE only"
assert float(
    CONTROL_CONTRACT["arms"]["control"]["alpha_human_ce"]
) == 1.0
assert float(CONTROL_CONTRACT["arms"]["control"]["alpha_kd"]) == 0.0
assert CONTROL_CONTRACT["test_set_status"] == "sealed"
assert CONTROL_CONTRACT["test_set_used"] is False

p2a_rows = verify_csv_manifest(
    P2A_PATH.parent, P2A["manifests"]["csv"]
)
assert len(p2a_rows) == 38
treatment_rows = verify_csv_manifest(
    TREATMENT_PATH.parent, TREATMENT["manifest"]["csv"]
)
assert len(treatment_rows) == 55

BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed42_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")

TREATMENT_STEP_BUDGETS = {
    42: 35420,
    123: 31878,
    2026: 30107,
}
assert int(TREATMENT["progress"]["global_optimizer_step"]) == (
    TREATMENT_STEP_BUDGETS[42]
)

write_json(AUDIT / "optimized_ce_cell01_authorization.json", {
    "schema_version": "optimized_ce_eta_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "seed": SEED,
    "gpu": torch.cuda.get_device_name(0),
    "student_initialization": "S1 seed-42 baseline-best",
    "objective": "human CE only",
    "teacher_model_required": False,
    "pseudo_caption_required": False,
    "vocabulary_mapping_required": False,
    "pilot_optimizer_steps": PILOT_STEPS,
    "pilot_weights_persisted": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("=" * 78)
print("OPTIMIZED CE CELL 1 — AUTHORIZATION")
print("=" * 78)
print("Status                    : PASSED")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Student init              : S1 seed-42 BASELINE-BEST")
print("Objective                 : HUMAN CE ONLY")
print("Teacher / pseudo / KL     : NOT REQUIRED")
print("Pilot steps               : 100")
print("Test Set status           : SEALED")


OPTIMIZED CE CELL 1 — AUTHORIZATION
Status                    : PASSED
GPU                       : Tesla T4
Student init              : S1 seed-42 BASELINE-BEST
Objective                 : HUMAN CE ONLY
Teacher / pseudo / KL     : NOT REQUIRED
Pilot steps               : 100
Test Set status           : SEALED


## Cell 2 — CE-only Train+Restval data path


In [2]:
STARTED = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision.transforms import InterpolationMode
from torchvision.transforms import RandomResizedCrop
from torchvision.transforms import functional as TF

TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
IMAGE_MANIFEST_PATH = exactly_one("mscoco_image_manifest.parquet")

def find_image_leaf(name, prefix):
    matches = []
    for path in DATASET_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]

TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("experiment_split", "=", "train")],
)
TRAIN_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = TRAIN_LEAF if row["source_partition"] == "train2014" else VAL_LEAF
    TRAIN_PATHS[image_id] = (leaf / Path(row["image_path"]).name).resolve()
assert len(TRAIN_PATHS) == 113287

TRAIN_CAPTIONS = defaultdict(list)
for arrow_batch in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in arrow_batch.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append((
            int(row["annotation_id"]),
            tuple(int(value) for value in row["token_ids"]),
        ))
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()
assert len(TRAIN_CAPTIONS) == 113287

TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|xlkd04"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8], "big"
    ) % (2 ** 63 - 1)

def student_train_transform(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = RandomResizedCrop.get_params(
            image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
        )
        do_flip = bool(torch.rand(()) < 0.5)
    student = TF.resized_crop(
        image, top, left, height, width, [224, 224],
        InterpolationMode.BICUBIC, antialias=True,
    )
    if do_flip:
        student = TF.hflip(student)
    return TF.normalize(
        TF.to_tensor(student), IMAGENET_MEAN, IMAGENET_STD
    )

class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, human_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        with Image.open(TRAIN_PATHS[image_id]) as source:
            student_image = student_train_transform(
                source.convert("RGB"), self.epoch, image_id
            )
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": student_image,
            "human_ids": torch.tensor(human_ids, dtype=torch.long),
        }


class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(self.length, generator=generator).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor

def production_collate(rows):
    assert len(rows) == 1
    row = rows[0]
    return {
        "image_ids": torch.tensor([row["image_id"]], dtype=torch.long),
        "annotation_ids": torch.tensor(
            [row["annotation_id"]], dtype=torch.long
        ),
        "student_image": row["student_image"].unsqueeze(0),
        "human_ids": row["human_ids"].unsqueeze(0),
    }


TRAIN_DATASET = ProductionTrainDataset()

def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    return DataLoader(
        TRAIN_DATASET,
        batch_size=1,
        sampler=EpochCursorSampler(len(TRAIN_DATASET), epoch, cursor),
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=production_collate,
    )

batch_a = next(iter(build_train_loader(0)))
batch_b = next(iter(build_train_loader(0)))
for key in [
    "image_ids", "annotation_ids", "student_image", "human_ids",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert batch_a["student_image"].shape == (1, 3, 224, 224)
assert torch.isfinite(batch_a["student_image"]).all()

write_json(AUDIT / "xlkd04_cell02_data_pipeline.json", {
    "schema_version": "xlkd04_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "train_images_per_epoch": len(TRAIN_DATASET),
    "human_caption_sampling": "one per image per epoch, cyclic",
    "teacher_sequence_per_image": 0,
    "physical_batch_size": 1,
    "student_stateless_geometry": True,
        "student_resolution": [224, 224],
    "kd_data_path_present": False,
    "two_loader_first_batch_exact": True,
    "resume_cursor_supported": True,
    "validation_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-03 CELL 2 — PRODUCTION DATA PIPELINE")
print("=" * 78)
print("Status                    : PASSED")
print("Train images / epoch      : 113,287")
print("Human caption / image     : 1 CYCLIC")
print("Teacher sequence / image  : 0 / REMOVED")
print("KD data path              : REMOVED")
print("Physical batch            : 1")
print("Student input             : 224")
print("Stateless augmentation    : True")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 2 / 7 code cells")


VALIDATION_PARQUET = exactly_one("mscoco_validation.parquet")
validation_path_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("experiment_split", "=", "validation")],
)
VALIDATION_PATHS = {}
for row in validation_path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = TRAIN_LEAF if row["source_partition"] == "train2014" else VAL_LEAF
    VALIDATION_PATHS[image_id] = (
        leaf / Path(row["image_path"]).name
    ).resolve()
assert len(VALIDATION_PATHS) == 5000

VALIDATION_RECORDS = []
VALIDATION_REFERENCES = defaultdict(list)
for arrow_batch in pq.ParquetFile(VALIDATION_PARQUET).iter_batches(
    batch_size=8192,
    columns=["annotation_id", "image_id", "token_ids", "caption"],
):
    for row in arrow_batch.to_pylist():
        image_id = int(row["image_id"])
        VALIDATION_RECORDS.append((
            int(row["annotation_id"]),
            image_id,
            tuple(int(value) for value in row["token_ids"]),
        ))
        VALIDATION_REFERENCES[image_id].append(row["caption"])
assert len(VALIDATION_RECORDS) == 25010
assert len(VALIDATION_REFERENCES) == 5000
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_PATHS))

def validation_transform(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    tensor = TF.to_tensor(TF.center_crop(image, [224, 224]))
    return TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)

class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_RECORDS)

    def __getitem__(self, index):
        annotation_id, image_id, token_ids = VALIDATION_RECORDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }

class ValidationImageDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {"image_id": image_id, "student_image": image}

def pad_sequences(sequences):
    maximum = max(item.numel() for item in sequences)
    result = torch.zeros(len(sequences), maximum, dtype=torch.long)
    for index, item in enumerate(sequences):
        result[index, :item.numel()] = item
    return result

def validation_caption_collate(rows):
    full = pad_sequences([row["token_ids"] for row in rows])
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "decoder_input_ids": full[:, :-1],
        "labels": full[:, 1:],
        "decoder_attention_mask": full[:, :-1].ne(0),
    }

def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
    }

VALIDATION_CAPTION_DATASET = ValidationCaptionDataset()
VALIDATION_IMAGE_DATASET = ValidationImageDataset()
VALIDATION_CAPTION_LOADER = DataLoader(
    VALIDATION_CAPTION_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=validation_caption_collate,
)
VALIDATION_IMAGE_LOADER = DataLoader(
    VALIDATION_IMAGE_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

assert set(VALIDATION_PATHS).isdisjoint(TRAIN_PATHS)
write_json(AUDIT / "xlkd04_cell02_validation_data.json", {
    "schema_version": "xlkd04_cell02_validation_data_v1",
    "created_at_utc": now(),
    "status": "passed",
    "training_images": len(TRAIN_DATASET),
    "validation_images": len(VALIDATION_IMAGE_DATASET),
    "validation_captions": len(VALIDATION_CAPTION_DATASET),
    "validation_overlap_with_training": 0,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Validation images         : 5,000")
print("Validation references     : 25,010")
print("Train/Validation overlap  : 0")
print("Test Set status           : SEALED")


XLKD-03 CELL 2 — PRODUCTION DATA PIPELINE
Status                    : PASSED
Train images / epoch      : 113,287
Human caption / image     : 1 CYCLIC
Teacher sequence / image  : 0 / REMOVED
KD data path              : REMOVED
Physical batch            : 1
Student input             : 224
Stateless augmentation    : True
Validation / Test         : NOT READ / SEALED
Progress                  : 2 / 7 code cells
Validation images         : 5,000
Validation references     : 25,010
Train/Validation overlap  : 0
Test Set status           : SEALED


## Cell 3 — Student-only optimizer engine


In [3]:
STARTED = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision.models import mobilenet_v3_large

DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")
decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080

class S1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ), diagonal=1
        )
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )

STUDENT_ENCODER = S1Encoder()
STUDENT_DECODER = SharedDecoder(decoder_config)
BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed42_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
baseline_payload = torch.load(
    BASELINE_CHECKPOINT, map_location="cpu", weights_only=False
)
assert baseline_payload["schema_version"] == (
    "phase2_s1_full_training_checkpoint_v1"
)
STUDENT_ENCODER.load_state_dict(
    baseline_payload["encoder_state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    baseline_payload["decoder_state_dict"], strict=True
)
BASE_ENCODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
BASE_DECODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
del baseline_payload
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 3231024
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5759328



ALPHA_HUMAN = 1.0
ALPHA_KD = 0.0
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 64
EFFECTIVE_BATCH = 64
STEPS_PER_EPOCH = math.ceil(113287 / EFFECTIVE_BATCH)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())
all_student_parameters = (
    backbone_parameters + adapter_parameters + decoder_parameters
)


def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    batch_norm_count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            batch_norm_count += 1
    assert batch_norm_count == 46


def reset_student_baseline():
    STUDENT_ENCODER.load_state_dict(BASE_ENCODER_STATE, strict=True)
    STUDENT_DECODER.load_state_dict(BASE_DECODER_STATE, strict=True)
    for parameter in all_student_parameters:
        parameter.grad = None


def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }


def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-4,
    )

    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / max(1, WARMUP_STEPS)
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler


def ce_only_optimizer_step(
    microbatches, optimizer, scheduler, scaler, retry_limit=10
):
    assert 1 <= len(microbatches) <= ACCUMULATION
    human_total = sum(
        int(batch["human_ids"][:, 1:].ne(0).sum())
        for batch in microbatches
    )
    assert human_total > 0
    retries = 0
    while True:
        set_student_train()
        optimizer.zero_grad(set_to_none=True)
        human_sum_value = 0.0
        for cpu_batch in microbatches:
            current = to_device(cpu_batch)
            human = current["human_ids"]
            with torch.amp.autocast("cuda", dtype=torch.float16):
                memory = STUDENT_ENCODER(current["student_image"])
                human_logits = STUDENT_DECODER(
                    human[:, :-1], memory, human[:, :-1].ne(0)
                )
                human_sum = F.cross_entropy(
                    human_logits.float().reshape(-1, 10080),
                    human[:, 1:].reshape(-1),
                    ignore_index=0,
                    reduction="sum",
                )
                normalized = human_sum / human_total
            scaler.scale(normalized / BACKWARD_DIVISOR).backward()
            human_sum_value += float(human_sum.detach())
            del (
                current,
                memory,
                human_logits,
                human_sum,
                normalized,
            )

        scaler.unscale_(optimizer)
        for parameter in all_student_parameters:
            if parameter.grad is not None:
                parameter.grad.mul_(BACKWARD_DIVISOR)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            all_student_parameters, 1.0
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        scheduler.step()
        human_ce = human_sum_value / human_total
        return {
            "combined_loss": human_ce,
            "human_ce": human_ce,
            "human_tokens": human_total,
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }


assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420

write_json(CONFIG / "optimized_ce_only_engine_contract.json", {
    "schema_version": "optimized_ce_only_engine_v1",
    "created_at_utc": now(),
    "status": "locked",
    "objective": "human CE only",
    "alpha_human_ce": ALPHA_HUMAN,
    "alpha_kd": ALPHA_KD,
    "teacher_loaded": False,
    "teacher_forward_calls_per_step": 0,
    "pseudo_decoder_forward_calls_per_step": 0,
    "kl_calls_per_step": 0,
    "mapped_vocabulary_operations_per_step": 0,
    "student_encoder_forward_calls_per_microbatch": 1,
    "student_decoder_forward_calls_per_microbatch": 1,
    "physical_batch": 1,
    "gradient_accumulation": ACCUMULATION,
    "effective_batch": EFFECTIVE_BATCH,
    "pilot_steps": PILOT_STEPS,
    "pilot_weights_persisted": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("=" * 78)
print("OPTIMIZED CE CELL 3 — ENGINE")
print("=" * 78)
print("Teacher loaded            : False")
print("Pseudo decoder forward    : 0")
print("KL / mapping operations   : 0 / 0")
print("Student enc/dec per micro : 1 / 1")
print("Physical/accum/effective  : 1 / 64 / 64")
print("Pilot steps               : 100")


OPTIMIZED CE CELL 3 — ENGINE
Teacher loaded            : False
Pseudo decoder forward    : 0
KL / mapping operations   : 0 / 0
Student enc/dec per micro : 1 / 1
Physical/accum/effective  : 1 / 64 / 64
Pilot steps               : 100


## Cell 4 — Pilot 100 optimizer step và ETA


In [4]:
CELL_STARTED_AT = time.perf_counter()
reset_student_baseline()
optimizer, scheduler, scaler = build_optimizer_state()
loader_iterator = iter(build_train_loader(epoch=0, cursor=0))
step_records = []
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

for pilot_step in range(1, PILOT_STEPS + 1):
    torch.cuda.synchronize()
    step_started = time.perf_counter()
    microbatches = [
        next(loader_iterator) for _ in range(ACCUMULATION)
    ]
    result = ce_only_optimizer_step(
        microbatches, optimizer, scheduler, scaler
    )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - step_started
    record = {
        "pilot_step": pilot_step,
        "elapsed_seconds": elapsed,
        **result,
        "peak_gpu_memory_mib": (
            torch.cuda.max_memory_allocated() / (1024 ** 2)
        ),
    }
    step_records.append(record)
    if pilot_step == 1 or pilot_step % 10 == 0:
        print(
            f"Pilot {pilot_step:03d}/{PILOT_STEPS}: "
            f"{elapsed:.3f} s/step | "
            f"CE={result['human_ce']:.6f} | "
            f"peak={record['peak_gpu_memory_mib']:.1f} MiB"
        )
    del microbatches, result

del loader_iterator
measured = np.asarray(
    [
        row["elapsed_seconds"]
        for row in step_records[PILOT_WARMUP_STEPS:]
    ],
    dtype=np.float64,
)
median_seconds = float(np.median(measured))
p90_seconds = float(np.percentile(measured, 90))
p95_seconds = float(np.percentile(measured, 95))
mean_seconds = float(np.mean(measured))
peak_memory_mib = float(max(
    row["peak_gpu_memory_mib"] for row in step_records
))
overflow_retries = int(sum(
    row["overflow_retries"] for row in step_records
))
stability_ratio = p95_seconds / max(median_seconds, 1e-12)

pilot_state_changed = any(
    not torch.equal(
        STUDENT_ENCODER.state_dict()[key].detach().cpu(), expected
    )
    for key, expected in BASE_ENCODER_STATE.items()
) or any(
    not torch.equal(
        STUDENT_DECODER.state_dict()[key].detach().cpu(), expected
    )
    for key, expected in BASE_DECODER_STATE.items()
)
reset_student_baseline()
state_restored = all(
    torch.equal(
        STUDENT_ENCODER.state_dict()[key].detach().cpu(), expected
    )
    for key, expected in BASE_ENCODER_STATE.items()
) and all(
    torch.equal(
        STUDENT_DECODER.state_dict()[key].detach().cpu(), expected
    )
    for key, expected in BASE_DECODER_STATE.items()
)

calendar_multiplier = 1.284
eta_rows = []
for seed, steps in TREATMENT_STEP_BUDGETS.items():
    step_hours_median = median_seconds * steps / 3600.0
    step_hours_p95 = p95_seconds * steps / 3600.0
    active_low = step_hours_median + 2.0
    active_high = step_hours_p95 + 5.0
    eta_rows.append({
        "seed": seed,
        "matched_optimizer_steps": steps,
        "step_only_hours_median": step_hours_median,
        "step_only_hours_p95": step_hours_p95,
        "active_gpu_hours_low": active_low,
        "active_gpu_hours_high": active_high,
        "calendar_days_low": active_low * calendar_multiplier / 24.0,
        "calendar_days_high": active_high * calendar_multiplier / 24.0,
    })

PILOT_PASSED = bool(
    len(step_records) == PILOT_STEPS
    and measured.size == PILOT_STEPS - PILOT_WARMUP_STEPS
    and np.isfinite(measured).all()
    and overflow_retries == 0
    and stability_ratio <= 1.50
    and peak_memory_mib < 14500.0
    and pilot_state_changed
    and state_restored
)
summary = {
    "schema_version": "optimized_ce_only_eta_pilot_v1",
    "created_at_utc": now(),
    "status": "passed" if PILOT_PASSED else "failed",
    "gpu": torch.cuda.get_device_name(0),
    "optimizer_steps": PILOT_STEPS,
    "warmup_steps_excluded": PILOT_WARMUP_STEPS,
    "seconds_per_optimizer_step": {
        "mean": mean_seconds,
        "median": median_seconds,
        "p90": p90_seconds,
        "p95": p95_seconds,
    },
    "p95_to_median_ratio": stability_ratio,
    "peak_gpu_memory_mib": peak_memory_mib,
    "overflow_retries": overflow_retries,
    "eta_by_matched_seed_budget": eta_rows,
    "calendar_multiplier": calendar_multiplier,
    "validation_packaging_allowance_hours": [2.0, 5.0],
    "teacher_loaded": False,
    "pseudo_decoder_forward_calls": 0,
    "kl_calls": 0,
    "mapping_operations": 0,
    "pilot_state_changed_before_restore": pilot_state_changed,
    "pilot_state_restored_exact": state_restored,
    "pilot_weights_persisted": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
write_json(RESULT / "optimized_ce_only_100step_eta.json", summary)
with (
    LOG / "optimized_ce_only_100step_steps.jsonl"
).open("w", encoding="utf-8") as handle:
    for row in step_records:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
write_json(AUDIT / "optimized_ce_cell04_eta_pilot.json", summary)

del optimizer, scheduler, scaler, step_records, measured
gc.collect()
torch.cuda.empty_cache()

print("=" * 78)
print("OPTIMIZED CE CELL 4 — 100-STEP ETA")
print("=" * 78)
print("Status                    :", summary["status"].upper())
print("Median / p95 s/step       :", median_seconds, "/", p95_seconds)
print("Peak GPU memory MiB       :", peak_memory_mib)
print("Overflow retries          :", overflow_retries)
for row in eta_rows:
    print(
        f"Seed {row['seed']} active h / calendar d:",
        f"{row['active_gpu_hours_low']:.2f}-"
        f"{row['active_gpu_hours_high']:.2f} / "
        f"{row['calendar_days_low']:.2f}-"
        f"{row['calendar_days_high']:.2f}",
    )
print("Pilot state restored      :", state_restored)
print("Pilot weights persisted   : False")
print("Test Set status           : SEALED")


Pilot 001/100: 14.173 s/step | CE=2.048967 | peak=177.3 MiB
Pilot 010/100: 3.551 s/step | CE=1.749401 | peak=207.9 MiB
Pilot 020/100: 3.436 s/step | CE=1.968002 | peak=208.0 MiB
Pilot 030/100: 3.420 s/step | CE=2.054603 | peak=208.0 MiB
Pilot 040/100: 3.470 s/step | CE=2.189575 | peak=208.0 MiB
Pilot 050/100: 3.400 s/step | CE=2.075126 | peak=208.0 MiB
Pilot 060/100: 3.423 s/step | CE=2.088342 | peak=208.0 MiB
Pilot 070/100: 3.450 s/step | CE=1.992180 | peak=208.0 MiB
Pilot 080/100: 3.462 s/step | CE=1.875165 | peak=208.0 MiB
Pilot 090/100: 3.561 s/step | CE=2.141641 | peak=208.0 MiB
Pilot 100/100: 3.453 s/step | CE=2.227536 | peak=208.0 MiB
OPTIMIZED CE CELL 4 — 100-STEP ETA
Status                    : PASSED
Median / p95 s/step       : 3.452945199999931 / 3.5537714251999772
Peak GPU memory MiB       : 207.9521484375
Overflow retries          : 0
Seed 42 active h / calendar d: 35.97-39.97 / 1.92-2.14
Seed 123 active h / calendar d: 32.58-36.47 / 1.74-1.95
Seed 2026 active h / calendar

## Cell 5 — Package và quyết định


In [5]:
CELL_STARTED_AT = time.perf_counter()
decision = {
    "schema_version": "optimized_ce_only_engine_decision_v1",
    "created_at_utc": now(),
    "status": "authorized" if PILOT_PASSED else "blocked",
    "optimized_engine_verified": True,
    "teacher_loaded": False,
    "pseudo_caption_data_loaded": False,
    "pseudo_decoder_forward_calls": 0,
    "kl_calls": 0,
    "mapping_operations": 0,
    "eta_pilot_passed": PILOT_PASSED,
    "authorized_next": (
        "Generate and run optimized matched CE controls for seeds 42/123/2026"
        if PILOT_PASSED
        else "Inspect failed ETA pilot gate before full control runs"
    ),
    "pilot_weights_persisted": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
marker_name = (
    "MATCHED_CE_OPTIMIZED_ENGINE_AUTHORIZED.json"
    if PILOT_PASSED
    else "MATCHED_CE_OPTIMIZED_ENGINE_BLOCKED.json"
)
MARKER_PATH = OUT / marker_name
write_json(MARKER_PATH, decision)

artifact_paths = []
for directory in [AUDIT, CONFIG, LOG, RESULT]:
    artifact_paths.extend(
        path for path in directory.rglob("*") if path.is_file()
    )
artifact_paths.append(MARKER_PATH)
artifact_paths = sorted(set(artifact_paths))
MANIFEST_CSV = MANIFEST / "optimized_ce_eta_manifest.csv"
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    for path in artifact_paths:
        writer.writerow({
            "relative_path": path.relative_to(OUT).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        })

with MANIFEST_CSV.open("r", encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))
for row in rows:
    artifact = resolve_inside(OUT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256(artifact) == row["sha256"]

complete = {
    **decision,
    "schema_version": "optimized_ce_only_eta_complete_v1",
    "manifest": {
        "relative_path": MANIFEST_CSV.relative_to(OUT).as_posix(),
        "sha256": sha256(MANIFEST_CSV),
        "rows": len(rows),
    },
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
COMPLETE_PATH = (
    OUT / "XLKD05_Dense_Mapped_LogitKD_"
    "OPTIMIZED_CE_100STEP_ETA_COMPLETE.json"
)
write_json(COMPLETE_PATH, complete)

print("=" * 78)
print("OPTIMIZED CE CELL 5 — PACKAGE")
print("=" * 78)
print("Status                    :", decision["status"].upper())
print("Manifest rows             :", len(rows))
print("Teacher / pseudo / KL     : REMOVED / REMOVED / REMOVED")
print("Pilot weights persisted   : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Marker                    :", MARKER_PATH)
print("Complete                  :", COMPLETE_PATH)


OPTIMIZED CE CELL 5 — PACKAGE
Status                    : AUTHORIZED
Manifest rows             : 8
Teacher / pseudo / KL     : REMOVED / REMOVED / REMOVED
Pilot weights persisted   : False
Full training started     : False
Test Set status           : SEALED
Marker                    : /kaggle/working//XLKD05_Dense_Mapped_LogitKD_optimized_ce_eta_pilot/MATCHED_CE_OPTIMIZED_ENGINE_AUTHORIZED.json
Complete                  : /kaggle/working//XLKD05_Dense_Mapped_LogitKD_optimized_ce_eta_pilot/XLKD05_Dense_Mapped_LogitKD_OPTIMIZED_CE_100STEP_ETA_COMPLETE.json
